# Personal AI Study Agent — Product & System Design Notebook

## Purpose
This notebook is a **professional design workbook** for planning a personal AI Study Agent as a realistic product prototype.

It is intended to guide structured decisions across:
- product scope
- user needs
- system architecture
- data design
- retrieval quality
- workflow orchestration
- evaluation
- delivery roadmap

This notebook is **not** a solution sheet.  
It is a decision framework for defining a system that is technically sound, practically useful, and feasible to implement.

---

## Working Principle
For each section, answer in a decision-oriented way:

1. **Decision**
2. **Reasoning**
3. **Trade-off / Risk**
4. **Next step**

Keep answers concise and concrete. Avoid vague statements such as:
- "we can decide later"
- "the model should handle this"
- "we may use an agent for everything"

The goal is to produce a design that could be defended in a project review, internship setting, or technical presentation.


## 1. Executive Product Definition

### Objective
Define the product before defining the technology.

### Task
Provide a concise product definition covering:

- target user
- primary problem
- expected business or practical value
- three core capabilities of the product
- clear non-goals
- success criteria for a first release

### Deliverable
End this section with:
- a **one-sentence product pitch**
- a **one-paragraph product summary**


# Executive Product Definition

**Target user:** University student managing 4–8 courses per semester, juggling lecture slides, handwritten notes, exercise sheets, and past exams across multiple subjects.

**Primary problem:** Students spend more time searching through disorganized material than actually learning. They lack a reliable way to know what to prioritize, what they already know, and what they have missed.

**Expected value:** Reduce time-to-understanding per topic by surfacing the right material at the right time. Replace manual re-reading with targeted, structured review.

**Three core capabilities:**
1. Semantic Q&A over all uploaded course materials
2. On-demand study plan generation based on course structure and exam proximity
3. Quiz and flashcard generation from indexed content

**Non-goals:**
- No collaborative features (multi-user, shared vaults)
- No real-time transcription of live lectures
- No autonomous agent that schedules or manages a calendar
- No general-purpose chatbot — only answers grounded in uploaded materials

**Success criteria for first release:**
- User can upload a PDF and ask a factual question and receive a correct, source-grounded answer within 5 seconds
- User can generate a study plan for one subject in under 30 seconds
- System correctly refuses to answer questions not covered by uploaded material

---

**One-sentence pitch:**
A personal AI study assistant that reads all your course materials and tells you exactly what to learn, when, and how.

**One-paragraph summary:**
The Personal AI Study Agent ingests a student's lecture slides, notes, exercise sheets, and past exams, indexes them into a searchable vector store, and provides a conversational interface for Q&A, summarization, study planning, and quiz generation. Unlike generic chatbots, every answer is grounded in the student's own uploaded materials. The system tracks which topics have been covered and adapts its recommendations accordingly. The initial release focuses on a single user, local storage, and a minimal chat-style interface — enough to be genuinely useful within one exam cycle.


## 2. Requirements Engineering

### Objective
Translate the idea into explicit requirements.

### Task
Define:
- **functional requirements**
- **non-functional requirements**
- **constraints**
- **assumptions**

Examples of relevant dimensions:
- correctness
- response quality
- speed
- transparency of answers
- maintainability
- privacy
- robustness on imperfect files

### Deliverable
Separate requirements into:
- Must-have
- Should-have
- Nice-to-have


# Requirements Engineering

## Functional Requirements

### Must-have
- Ingest PDF, PPTX, and plain text files
- Chunk, embed, and index content into a vector store on upload
- Answer factual questions using retrieved content (RAG)
- Return source references with every answer (document name, page or slide number)
- Summarize a single document or a set of documents on request
- Generate a prioritized study plan per subject on request
- Generate multiple-choice or open-ended quiz questions from indexed content

### Should-have
- Filter retrieval by course or document type (slides, exercises, exams)
- Detect and flag duplicate or conflicting content across documents
- Export flashcards in Anki-compatible format (CSV or via AnkiConnect)
- Store quiz results and track which topics were answered correctly
- Support reupload and incremental update of modified documents

### Nice-to-have
- Obsidian markdown export for generated summaries and study plans
- OCR support for scanned PDFs and handwritten note images
- Weak-topic detection based on quiz history
- Cross-document timeline of when topics were introduced

---

## Non-functional Requirements

### Must-have
- Answer latency under 8 seconds for standard Q&A queries
- All data stored locally by default — no content sent to third-party services beyond the LLM API call
- System must not hallucinate: answers must be refusable when no relevant context is found

### Should-have
- Graceful degradation on malformed or image-heavy PDFs
- Clear error messages when a file cannot be parsed
- Embedding index must survive application restarts (persistent vector store)

### Nice-to-have
- Response streaming for long answers
- Configurable chunk size and overlap per course

---

## Constraints
- Solo developer, part-time build
- Budget: minimal — OpenAI or Anthropic API costs only; no paid infrastructure
- Runs locally on a personal laptop (macOS or Linux)
- No authentication or multi-user logic required

## Assumptions
- User uploads well-formed PDFs or PPTX files in most cases
- One user, one local machine
- Exam calendar is manually entered or inferred from document names
- Documents are in German or English; multilingual support is not a requirement but answers should match the language of the question


## 3. MVP Scope and Release Boundaries

### Objective
Prevent overengineering and clarify what belongs into the first usable version.

### Task
Define what belongs to:
- **v1 (MVP)**
- **v2**
- **later / optional**

Consider at least:
- document Q&A
- summarization
- study plan generation
- old-exam analysis
- quiz / flashcard generation
- progress tracking
- Obsidian integration
- Anki integration
- multi-step agent workflows

### Deliverable
Provide one final sentence:
**Why is your MVP small enough to build but large enough to be useful?**


# MVP Scope and Release Boundaries

## v1 — MVP
- PDF and PPTX upload and parsing
- Chunking, embedding (OpenAI text-embedding-3-small), and indexing into ChromaDB
- Metadata per chunk: source file, course name, document type, page number
- Document Q&A with source references
- Single-document and multi-document summarization
- Basic study plan generation (LLM-generated, based on retrieved topic list)
- Streamlit chat interface with file upload sidebar
- Local persistence (ChromaDB on disk + SQLite for metadata)
- Refusal when no relevant context is found

## v2
- Quiz and flashcard generation (in-app + CSV export for Anki)
- Course-level filtering in queries ("only search my Analysis slides")
- Quiz result tracking and weak-topic flagging
- Old-exam analysis ("what topics appear most in past exams?")
- Incremental re-indexing on file update
- Obsidian markdown export for summaries and plans

## Later / Optional
- OCR for scanned PDFs and handwritten notes
- AnkiConnect live sync
- Progress dashboard with topic coverage visualization
- Multi-subject exam prioritization across courses
- Automated document watching from a local folder

---

**Why is this MVP small enough to build but large enough to be useful?**
It covers the single most painful moment in a student's workflow — "I have all this material, where do I start?" — without requiring any integrations, cloud infrastructure, or advanced agent behavior. A student with one exam in two weeks can immediately benefit from Q&A and a generated study plan using only their existing slides.


## 4. User Scenarios and Primary Workflows

### Objective
Design the system around actual usage rather than abstract capabilities.

### Task
Describe at least 3 realistic user scenarios, for example:
- a user uploads a new lecture and asks what matters most
- a user wants a summary across slides, notes, and exercise sheets
- a user wants a quiz on weak topics before an exam

For each scenario, define:
- trigger
- user input
- internal system actions
- output
- what should be stored afterwards

### Deliverable
Explain which workflow is the most important for launch and why.


# User Scenarios and Primary Workflows

## Scenario 1 — New lecture uploaded, student wants to know what matters

**Trigger:** Student uploads this week's lecture slides after class.
**User input:** "What are the most important concepts in today's lecture?"
**Internal actions:**
1. File is parsed and chunked on upload
2. Chunks are embedded and added to ChromaDB with metadata: course="Algorithms", type="lecture", date="2025-04-16"
3. Query is embedded and top-k relevant chunks are retrieved
4. LLM generates a prioritized topic list with brief explanations, grounded in retrieved content
**Output:** Bullet list of key concepts with source references (slide numbers)
**What is stored:** Processed file, chunks, embeddings, document metadata in SQLite

---

## Scenario 2 — Cross-material summary before the exam

**Trigger:** Student has an exam in 5 days and wants a consolidated summary.
**User input:** "Summarize everything I have for the Algorithms course."
**Internal actions:**
1. Query filters ChromaDB by course="Algorithms"
2. All chunks for the course are retrieved (or top-k per document if too many)
3. LLM generates a structured summary grouped by topic
4. Summary is stored in SQLite with timestamp and course tag
**Output:** Structured markdown summary (topics, definitions, key formulas)
**What is stored:** Generated summary saved for reuse and future comparison

---

## Scenario 3 — Quiz on weak topics before the exam

**Trigger:** Student wants to self-test before the exam.
**User input:** "Give me 10 quiz questions on graph algorithms."
**Internal actions:**
1. Query retrieves chunks tagged with relevant topic keywords
2. LLM generates multiple-choice questions with distractors and correct answers
3. Student answers in the UI; results are recorded per question
**Output:** Interactive quiz in the UI; correct answers shown after each response
**What is stored:** Quiz session: questions, user answers, scores, topic tags — used for weak-topic detection in v2

---

**Most important workflow for launch:** Scenario 1 (new lecture Q&A). It is the highest-frequency action — happens after every class — and immediately demonstrates value with zero setup beyond uploading one file. If this works reliably, students will keep uploading material.


## 5. Input Sources and Data Quality Assessment

### Objective
Identify which sources the system should use and how reliable they are.

### Task
List all relevant input sources, for example:
- lecture slides
- lecture notes
- exercise sheets
- old exams
- personal notes
- handwritten scans
- generated summaries
- Obsidian notes

For each source, assess:
- file format
- expected data quality
- parsing difficulty
- importance
- whether it belongs in v1

### Deliverable
Highlight the three most critical source types for early product value.


# Input Sources and Data Quality Assessment

| Source | Format | Data Quality | Parsing Difficulty | Importance | v1? |
|---|---|---|---|---|---|
| Lecture slides | PDF / PPTX | Medium — often text-light, image-heavy | Medium | Critical | Yes |
| Lecture notes (typed) | PDF / MD / TXT | High | Low | High | Yes |
| Exercise sheets | PDF | Medium — may include formulas, tables | Medium | High | Yes |
| Old exams | PDF | Medium — mixed text and formatting | Medium | High | Yes |
| Personal handwritten notes | PNG / scanned PDF | Low — OCR required | High | Medium | No |
| Generated summaries | Markdown | High | Trivial | Medium | Yes (v2 auto-ingest) |
| Obsidian notes | Markdown | High | Low | Low–Medium | No |

**Parsing notes:**
- PPTX files can be parsed with `python-pptx` to extract text per slide with slide number metadata
- Image-only PDFs (scanned slides, photos of boards) will yield no text — OCR is deferred to v2
- PDFs with embedded formulas (LaTeX-rendered images) will lose the formula content in v1 — acceptable limitation

---

**Three most critical source types for early product value:**

1. **Lecture slides (PDF/PPTX)** — highest information density per course; the primary source students study from
2. **Old exams (PDF)** — directly inform study priority; which topics get examined is the most actionable signal
3. **Typed lecture notes (PDF/TXT)** — high text quality, easy to parse, fills gaps in slides


## 6. Ingestion and Indexing Architecture

### Objective
Define what happens from file upload to retrievable knowledge.

### Task
Design the ingestion pipeline end to end:
- upload
- file parsing
- OCR where needed
- cleaning / normalization
- chunking
- metadata extraction
- embedding
- indexing
- update handling
- failure handling

### Deliverable
Describe the pipeline in two forms:
1. step-by-step bullet flow
2. one concise technical narrative


# Ingestion and Indexing Architecture

## Step-by-step pipeline

1. **Upload** — User selects one or more files via the Streamlit sidebar; files are written to a local `data/raw/<course>/` folder
2. **File type detection** — Extension-based routing: `.pdf` → PyMuPDF, `.pptx` → python-pptx, `.txt`/`.md` → direct read
3. **Parsing** — Text extracted per page (PDF) or per slide (PPTX); slide/page number recorded as metadata
4. **OCR fallback** — v1: if a PDF page yields fewer than 30 characters, log a warning and skip; no OCR in v1
5. **Cleaning** — Strip headers, footers, and repeated boilerplate using heuristics (line length, repetition across pages); normalize whitespace
6. **Chunking** — Recursive character text splitter; chunk size 512 tokens, overlap 64 tokens
7. **Metadata extraction** — Per chunk: `source_file`, `course`, `doc_type` (slide/note/exam/exercise), `page_number`, `ingestion_timestamp`
8. **Embedding** — OpenAI `text-embedding-3-small` via batch API call; 1536 dimensions
9. **Indexing** — Chunks + embeddings + metadata written to ChromaDB collection named after the course
10. **Deduplication** — Before writing, check chunk hash against stored hashes in SQLite; skip if already present
11. **Metadata persistence** — Document record (filename, course, doc_type, page count, status) written to SQLite `documents` table
12. **Failure handling** — Per-document try/except; failed files logged to SQLite with error reason; user notified in UI; system continues with remaining files

---

**Technical narrative:**
When a file is uploaded, the system routes it to the appropriate parser based on file type, extracts text with structural metadata (page or slide number), cleans it, and splits it into overlapping token-bounded chunks. Each chunk is hashed for deduplication, embedded using the OpenAI embedding API, and written to a persistent ChromaDB collection keyed by course. A SQLite record tracks every processed document and every ingestion failure. The entire pipeline runs synchronously in v1; files larger than 200 pages may take 10–30 seconds and should display a progress indicator.


## 7. Retrieval Strategy and Knowledge Access

### Objective
Ensure the system retrieves the right information instead of merely searching approximate text.

### Task
Define:
- chunking strategy
- overlap policy
- metadata schema
- retrieval strategy
- top-k selection
- filtering rules
- optional reranking
- answer grounding strategy

Also address:
- duplicate content
- conflicting sources
- scanned material
- mixed file types

### Deliverable
State how your design reduces hallucinations and irrelevant retrieval.


# Retrieval Strategy and Knowledge Access

## Chunking strategy
- Recursive character splitter with semantic boundary preference (paragraph → sentence → character)
- Chunk size: 512 tokens, overlap: 64 tokens
- Justification: 512 tokens fits comfortably within a typical LLM context window even when 8–12 chunks are concatenated; 64-token overlap prevents concept splits at boundaries

## Metadata schema (per chunk)
```
{
  "chunk_id": "uuid",
  "source_file": "Analysis_Lecture_04.pdf",
  "course": "Analysis",
  "doc_type": "lecture|exercise|exam|note",
  "page_number": 12,
  "ingestion_timestamp": "2025-04-16T10:00:00",
  "chunk_hash": "sha256..."
}
```

## Retrieval strategy
- Primary: dense vector similarity search (cosine) in ChromaDB
- Filter by `course` metadata before similarity ranking — prevents cross-course contamination
- Optional filter by `doc_type` when the user specifies ("only look in old exams")
- Top-k: 8 chunks per query (tunable)

## Reranking
- v1: no dedicated reranker — top-k by cosine similarity is sufficient for single-course scopes
- v2: add Cohere rerank or cross-encoder if retrieval quality is poor on longer documents

## Answer grounding
- All LLM prompts include a strict instruction: "Answer only based on the provided context. If the answer is not in the context, say so explicitly."
- Every answer includes source citations: file name + page/slide number
- If retrieved chunks score below a cosine similarity threshold (e.g. 0.30), the system returns a soft refusal

## Handling edge cases
- **Duplicates:** Chunk hash deduplication at ingestion prevents identical content from inflating retrieval scores
- **Conflicting sources:** LLM instructed to note when retrieved chunks contradict each other; user sees both sources
- **Scanned material:** Skipped in v1 with a clear user warning on upload
- **Mixed file types:** Handled by metadata filtering; doc_type allows targeted retrieval

---

**How this design reduces hallucinations:**
The combination of (1) strict grounding instructions in the system prompt, (2) cosine similarity thresholding for soft refusals, and (3) source citations in every answer means the LLM cannot plausibly invent content without the student immediately noticing the missing source reference. Retrieval failures surface as "no relevant content found" rather than fabricated answers.


## 8. Storage Design

### Objective
Separate the different storage responsibilities clearly.

### Task
Define where and how you will store:
- raw files
- parsed text
- chunks and embeddings
- metadata
- user progress
- generated summaries
- quiz history
- exports to external tools

Take a position on:
- local vs cloud
- Chroma vs Supabase pgvector
- SQL needs
- persistence and maintainability

### Deliverable
Provide a final storage architecture decision with justification.


# Storage Design

## Storage responsibilities

| Data type | Store | Format | Notes |
|---|---|---|---|
| Raw uploaded files | Local filesystem | Original files | `data/raw/<course>/<filename>` |
| Parsed text (intermediate) | Not persisted | In-memory only | Re-parseable from raw files on demand |
| Chunks + embeddings | ChromaDB (local) | Native Chroma format | Persistent on disk; one collection per course |
| Chunk metadata | ChromaDB metadata fields | Key-value per chunk | Queried alongside vectors |
| Document registry | SQLite | `documents` table | Tracks ingestion status, file hash, page count |
| Generated summaries | SQLite | `summaries` table | Stored with course, timestamp, doc scope |
| Quiz history | SQLite | `quiz_sessions`, `quiz_answers` tables | Per question: topic, user answer, correct/wrong |
| User progress / weak topics | SQLite | `topic_scores` table | Computed from quiz history |
| Anki export | Local filesystem | CSV | `exports/anki/<course>_<date>.csv` |
| Obsidian export (v2) | Local filesystem | Markdown | `exports/obsidian/<course>/` |

## Architecture decision: local-first

**Decision:** ChromaDB on disk + SQLite — no cloud storage in v1.

**Reasoning:**
- Zero cost, zero network dependency, zero privacy risk
- ChromaDB persists to a local directory with a single config line
- SQLite requires no server and is sufficient for one user's data volume
- Migrating to Supabase (pgvector + PostgreSQL) in v2 is straightforward if cloud access becomes necessary

**Trade-off:** No cross-device access. Acceptable for v1 — the student runs the app on one machine.

**Supabase pgvector considered and deferred:** pgvector is the right choice if cloud sync or web hosting is needed. Chroma's simpler API is faster to implement and sufficient for local use.

---

**Final storage architecture:**
One ChromaDB instance (local, persistent) for vectors and chunk metadata. One SQLite database for all relational data (documents, summaries, quiz history, progress). All raw files kept on disk as the authoritative source. Exports written to a local `exports/` folder. Total storage footprint for one student's full semester: estimated 50–200 MB.


## 9. LLM Responsibilities and Model Selection

### Objective
Prevent poor system design by assigning too much logic to the LLM.

### Task
Define explicitly:
- what the LLM is responsible for
- what should be handled outside the LLM
- where deterministic logic is required
- where tool calls are needed

Then choose:
- primary model
- optional secondary model
- embedding model

### Deliverable
Conclude with a short rationale:
**Why is this model setup appropriate for the product stage and budget?**


# LLM Responsibilities and Model Selection

## What the LLM is responsible for
- Generating answers from retrieved context (RAG completion)
- Summarizing sets of retrieved chunks into coherent prose
- Generating a structured study plan from a topic list
- Generating quiz questions and answer options from content chunks
- Detecting contradictions between retrieved chunks
- Deciding when retrieved context is insufficient (soft refusal)

## What is handled outside the LLM
- Document parsing, chunking, and embedding (deterministic pipeline)
- Retrieval and filtering from ChromaDB (vector similarity + metadata filter)
- Deduplication via chunk hashing
- Storage reads and writes (SQLite)
- File routing by type
- Cosine similarity thresholding for refusal trigger
- Quiz result scoring (string match / answer key comparison)
- Export formatting (Anki CSV, Markdown)

## Where deterministic logic is required
- Deduplication (hash-based, not LLM-based)
- Retrieval ranking (cosine similarity, not LLM judgment)
- Score computation (correct / incorrect answers)
- Export formatting

## Tool calls (v2)
- Anki export trigger ("generate and export flashcards as Anki CSV")
- Obsidian write trigger ("save this summary to my Obsidian vault")

---

## Model selection

| Role | Model | Justification |
|---|---|---|
| Primary LLM | GPT-4o (via OpenAI API) | Strong instruction following, reliable grounding behavior, structured output support |
| Optional fallback | Claude Haiku 4.5 | Faster and cheaper for simple Q&A with short context |
| Embedding model | text-embedding-3-small | 1536 dimensions, low cost (~$0.02 per 1M tokens), strong retrieval performance for academic text |

---

**Why is this model setup appropriate for the product stage and budget?**
GPT-4o provides reliable grounding and structured output without requiring fine-tuning or complex prompting. `text-embedding-3-small` is the most cost-efficient embedding model available and handles academic text in both German and English well. For a single student's usage volume (tens of queries per day), total API cost is under €5/month — well within a personal project budget. The setup requires no model hosting, no GPU, and no DevOps.


## 10. Functional Modes of the Study Agent

### Objective
Turn the product into a set of clear functional services instead of one overloaded chat assistant.

### Task
Define the main operating modes:
- Q&A mode
- summarization mode
- study plan mode
- exam analysis mode
- quiz / flashcard mode

For each mode, specify:
- input
- required sources
- internal logic
- output format
- key risk or failure mode

### Deliverable
State which mode is the anchor feature for v1.


# Functional Modes of the Study Agent

## Q&A Mode
**Input:** Free-text question from the user
**Required sources:** All indexed chunks for the selected course (or all courses if unfiltered)
**Internal logic:** Embed query → retrieve top-8 chunks → filter by cosine threshold → LLM generates grounded answer with citations
**Output format:** Prose answer + source list (file name + page number)
**Key risk:** Retrieval misses the relevant chunk if the question uses different vocabulary than the source text → mitigated by generous overlap and semantic embedding

## Summarization Mode
**Input:** "Summarize [course/document/topic]"
**Required sources:** All chunks for the specified scope
**Internal logic:** Retrieve top-k chunks by relevance or retrieve all chunks filtered by metadata → LLM generates structured summary with topic headers
**Output format:** Markdown with headers, bullet points for key definitions, bold terms
**Key risk:** Very long courses exceed context window → mitigated by hierarchical summarization (summarize per document, then summarize summaries)

## Study Plan Mode
**Input:** "Create a study plan for [course], exam in [N] days"
**Required sources:** All document titles and topic lists for the course; old exam doc_type chunks if available
**Internal logic:** Retrieve topic inventory → LLM identifies high-frequency exam topics → generates prioritized day-by-day plan
**Output format:** Structured list: Day → Topics → Recommended materials (source references)
**Key risk:** LLM invents topics not present in the material → mitigated by grounding the plan strictly in retrieved topic names

## Exam Analysis Mode
**Input:** "What topics appear most in my old exams for [course]?"
**Required sources:** All chunks where doc_type="exam"
**Internal logic:** Retrieve all exam chunks → LLM identifies and counts recurring topic clusters → returns ranked topic list
**Output format:** Ranked list with topic name and estimated frequency
**Key risk:** Sparse old exam material (only 1–2 files) gives unreliable frequency signals → user is informed of the sample size

## Quiz / Flashcard Mode
**Input:** "Quiz me on [topic]" or "Generate flashcards for [course]"
**Required sources:** Chunks relevant to the specified topic or course
**Internal logic:** Retrieve relevant chunks → LLM generates questions with distractors (MC) or answer pairs (flashcard) → results stored per session
**Output format:** In-app interactive quiz or exportable CSV (Anki-compatible)
**Key risk:** LLM generates questions that are too easy or trivially answered by surface pattern matching → partially mitigated by prompt instructions requiring concept-level questions

---

**Anchor feature for v1:** Q&A Mode. It requires the fewest components (RAG pipeline only), delivers immediate value after the first file upload, and is the usage pattern that builds user trust. All other modes depend on the same retrieval infrastructure, so Q&A mode validates the core stack before investing in additional output formats.


## 11. Memory, Personalization, and Progress Tracking

### Objective
Explain how the system evolves from a document chatbot into a real personal study assistant.

### Task
Define what user-related state should be stored, for example:
- processed files
- completed topics
- weak topics
- past summaries
- quiz performance
- revision priorities
- user preferences

For each category, explain:
- why it matters
- where it is stored
- whether it affects future retrieval or generation

### Deliverable
Answer this directly:
**What minimum memory is required so that the product feels personalized?**


# Memory, Personalization, and Progress Tracking

## User state to store

### Processed files
- **Why:** Prevents re-ingestion; enables incremental updates; shows the user what the system knows
- **Stored in:** SQLite `documents` table (filename, course, doc_type, ingestion_date, page_count, status)
- **Affects retrieval:** Directly — only ingested documents are searchable

### Completed topics
- **Why:** Enables the study plan to skip topics the student has already reviewed
- **Stored in:** SQLite `topic_progress` table (topic_name, course, marked_complete, timestamp)
- **Affects generation:** Study plan mode filters out completed topics when generating recommendations

### Weak topics
- **Why:** The most actionable personalization signal — tells the student where to focus
- **Stored in:** Computed from `quiz_answers` table: topic + (wrong_answers / total_answers) → weakness score
- **Affects retrieval:** Quiz mode prioritizes chunks for weak topics; study plan mode bumps weak topics up in priority

### Past summaries
- **Why:** Avoids regenerating the same summary; enables comparison over time
- **Stored in:** SQLite `summaries` table (course, scope, content_markdown, created_at)
- **Affects generation:** Shown as cached output when the same scope is requested again

### Quiz performance
- **Why:** Primary signal for weak-topic detection; also motivating for the student
- **Stored in:** SQLite `quiz_sessions` and `quiz_answers` tables (session_id, question, user_answer, correct, topic_tag)
- **Affects retrieval:** Topic weakness scores are derived from this table

### Revision priorities
- **Why:** Allows the student to manually flag topics for focused review
- **Stored in:** SQLite `revision_flags` (topic, course, priority_level, created_at)
- **Affects generation:** Study plan mode incorporates manual flags alongside computed weakness

### User preferences
- **Why:** Preferred answer language, default course filter, chunk size — small QoL improvements
- **Stored in:** SQLite `settings` table (key, value)
- **Affects retrieval/generation:** Language preference passed in system prompt; default course filter applied at query time

---

**Minimum memory required for the product to feel personalized:**
The system must remember (1) which files have been uploaded per course, and (2) which topics the student got wrong in quizzes. With just these two signals, the study plan can say "you haven't reviewed Chapter 3 and you scored poorly on graph traversal — start there." Everything else is an enhancement.


## 12. Role of Obsidian and External Tools

### Objective
Place external tools in the architecture correctly.

### Task
Decide the role of:
- Obsidian
- Anki / AnkiConnect
- optional notebook-style tools

For Obsidian, answer specifically:
- is it output only, input only, or both?
- what content is written there?
- when is content synced?
- why is it not the system backend?

### Deliverable
Summarize the value of each external tool in one sentence.


# Role of Obsidian and External Tools

## Obsidian
- **Role:** Output only in v1; bidirectional in v2 (Obsidian notes as an input source)
- **What content is written there:** Generated summaries and study plans, exported as Markdown files into the Obsidian vault folder
- **When content is synced:** On explicit user action ("Export to Obsidian") — not automatically
- **Why it is not the system backend:** Obsidian is a Markdown file viewer with a plugin ecosystem, not a queryable database. Using it as a backend would require the Dataview plugin and impose a fragile file-naming convention on the indexing layer. The vector store (ChromaDB) is the system's source of truth; Obsidian is a human-readable view layer.

**Value in one sentence:** Obsidian gives the student a permanent, readable, linkable record of AI-generated summaries and study plans inside their existing knowledge management workflow.

---

## Anki / AnkiConnect
- **Role:** Export target for generated flashcards
- **v1:** Export as Anki-importable CSV (semicolon-delimited, front/back columns) — no live connection required
- **v2:** AnkiConnect REST API for live deck creation and card push without manual import
- **Why not in-app only:** Anki's spaced repetition algorithm and mobile app are mature, proven tools. Replicating SRS in-app would be over-engineering for v1.

**Value in one sentence:** Anki handles the spaced repetition scheduling so the study agent only needs to generate the card content, not manage review intervals.

---

## NotebookLM (optional, not integrated)
- **Role:** Not integrated in v1 — no public API available
- **Pragmatic approach:** The agent exports clean Markdown or PDF summaries that the student manually uploads to NotebookLM for audio overviews and NotebookLM-native flashcards
- **v2 consideration:** If Google releases a NotebookLM API, it could be a target for automated summary delivery

**Value in one sentence:** NotebookLM adds audio-format review and a second AI perspective on the same material, used as a manual downstream consumer of the agent's exports rather than an integrated component.


## 13. Interface and User Experience

### Objective
Define the minimum interface needed for a usable first release.

### Task
Choose the interface approach:
- Streamlit
- FastAPI + frontend
- notebook-first prototype
- other

Then define:
- minimum UI features
- upload experience
- result presentation
- source transparency
- export options
- future UI improvements

### Deliverable
Explain why this interface is appropriate for a first product version.


# Interface and User Experience

## Interface choice: Streamlit

**Decision:** Streamlit for v1.

**Reasoning:** Streamlit requires no frontend development skills, runs locally with `streamlit run app.py`, and can render chat messages, file upload widgets, markdown output, and source references in under 200 lines of Python. It is the fastest path to a working, usable interface for a solo developer.

---

## Minimum UI features for v1

- **Sidebar:** Course selector (dropdown), file uploader (multi-file, PDF/PPTX/TXT), ingestion status indicator per file
- **Chat interface:** Chat history (using `st.chat_message`), text input for queries, mode selector (Q&A / Summarize / Study Plan / Quiz)
- **Answer display:** LLM response in chat bubble, source citations listed below each answer (filename + page number)
- **Study plan view:** Rendered as a structured markdown block with collapsible sections per day
- **Quiz view:** One question at a time, multiple-choice buttons, score shown after each answer

## Upload experience
- Drag-and-drop or file picker in the sidebar
- Per-file progress bar during ingestion (parsing → embedding → indexing)
- Success/failure status badge per file
- Clear warning if a file produced no extractable text (likely scanned)

## Result presentation
- Answers rendered in Markdown (headers, bold, bullet points)
- Source references as a collapsible "Sources" section beneath the answer
- If the system cannot answer (below similarity threshold), a clear message: "No relevant content found in your uploaded materials."

## Source transparency
- Every answer includes file name + page/slide number for each retrieved chunk used
- Optional: show retrieved chunk snippets on hover or in an expandable section

## Export options
- "Export as Markdown" button for summaries and study plans (downloads a `.md` file)
- "Export to Anki CSV" button in quiz/flashcard mode (downloads a `.csv` file)

## Future UI improvements (v2+)
- Course-level dashboard: ingested documents, topic coverage map, quiz score history
- Weak topic heatmap
- Side-by-side view of source document and generated summary
- FastAPI + React frontend for a more polished product feel

---

**Why this interface is appropriate for a first product version:**
Streamlit lets the entire product be built and iterated by one developer without context-switching to frontend work. The interface is functional, not beautiful — but it surfaces every core capability (upload, Q&A, summarize, quiz) in a form a student can actually use during an exam period. A polished UI is a v2 concern; correct, trustworthy answers are a v1 requirement.


## 14. System Architecture Overview

### Objective
Consolidate all design choices into one coherent system architecture.

### Task
Describe the architecture across four layers:
1. ingestion
2. storage
3. agent core
4. output / interface

Then write one end-to-end request flow:
- user uploads material
- system processes and stores it
- user asks a question
- retrieval happens
- answer is generated
- outputs and state are persisted

### Deliverable
Write the final architecture summary as if you were explaining it in a technical review.


# System Architecture Overview

## Four layers

### 1. Ingestion layer
- File upload via Streamlit sidebar
- Router: PyMuPDF (PDF), python-pptx (PPTX), plain text reader (TXT/MD)
- Cleaning + chunking (RecursiveCharacterTextSplitter, 512 tokens, 64 overlap)
- Embedding via OpenAI `text-embedding-3-small`
- Deduplication via SHA-256 chunk hash checked against SQLite
- Write to ChromaDB (vectors + metadata) and SQLite (document registry)

### 2. Storage layer
- ChromaDB (local, persistent): one collection per course; stores chunk text, embeddings, and metadata
- SQLite: documents table, summaries table, quiz_sessions/quiz_answers tables, topic_progress table, settings table
- Local filesystem: raw uploaded files (`data/raw/`), exports (`exports/`)

### 3. Agent core
- Query router: classifies user input into one of five modes (Q&A, summarize, study plan, exam analysis, quiz)
- Retrieval: ChromaDB cosine similarity search with metadata filters; top-8 chunks; similarity threshold check
- LLM call: GPT-4o with mode-specific system prompt; context = concatenated retrieved chunks; strict grounding instruction
- Post-processing: extract source citations, check for refusal condition, format output per mode
- State write: store summaries, quiz results, and topic progress back to SQLite

### 4. Output / interface layer
- Streamlit chat UI renders LLM response as Markdown
- Source citations displayed beneath each answer
- Mode-specific UI: quiz buttons, plan view, export buttons
- Export: Markdown file download, Anki CSV download

---

## End-to-end request flow

1. **User uploads "Analysis_Lecture_05.pdf"**
   → File written to `data/raw/Analysis/`
   → PyMuPDF extracts text per page
   → Chunks created (512 tokens, 64 overlap)
   → Each chunk hashed; duplicates skipped
   → Chunks embedded via OpenAI API (batch call)
   → Chunks written to ChromaDB collection "Analysis" with metadata
   → Document record written to SQLite

2. **User asks: "What is the definition of uniform continuity?"**
   → Input classified as Q&A mode
   → Query embedded via OpenAI API
   → ChromaDB queried: top-8 chunks from collection "Analysis", cosine similarity > 0.30
   → Retrieved chunks concatenated into context
   → GPT-4o called with system prompt (grounding instruction) + context + user question
   → Response parsed; source citations extracted
   → Answer displayed in Streamlit chat with sources listed below

3. **State persisted:** No state written for a plain Q&A query. If the user requests a summary, the result is stored in SQLite. If a quiz answer is submitted, the result is recorded.

---

**Architecture summary for a technical review:**
The system is a straightforward RAG pipeline with a local-first storage model and a mode-switched LLM layer. ChromaDB handles vector search; SQLite handles all relational state. The LLM is responsible only for text generation — all retrieval, deduplication, scoring, and export logic is deterministic. The Streamlit interface is a thin wrapper over the Python backend. The architecture is intentionally simple: no agent loops, no tool-calling framework, no cloud infrastructure. This makes it fast to build, easy to debug, and reliable enough for real exam-period use. Complexity is introduced incrementally in v2 through targeted additions (reranking, weak-topic logic, AnkiConnect) rather than upfront.


## 15. Risks, Limitations, and Mitigation

### Objective
Show engineering maturity by identifying the main failure points early.

### Task
List the top risks in your system, for example:
- poor OCR quality
- weak retrieval
- hallucinated answers
- duplicated or outdated notes
- wrong study priorities
- user trust issues
- scaling and latency

For each risk, define:
- impact
- likelihood
- mitigation strategy

### Deliverable
Identify the single highest-priority risk for v1.


# Risks, Limitations, and Mitigation

## Risk register

| Risk | Impact | Likelihood | Mitigation |
|---|---|---|---|
| Poor PDF text extraction (scanned, image-heavy slides) | High — large portions of material not indexed | High — many lecture slides use image-based layouts | Warn user on upload if < 30 chars/page extracted; defer OCR to v2 |
| Weak retrieval (wrong chunks returned) | High — answers are irrelevant or missing key content | Medium | Tune chunk size and overlap; add cosine threshold; test retrieval with known Q&A pairs |
| LLM hallucination | High — student studies incorrect information | Low with grounding prompt | Strict grounding instruction; source citations; cosine threshold refusal; user-visible sources enable fact-checking |
| Duplicate or outdated notes | Medium — retrieval returns stale content | Medium — students re-upload revised files | Chunk hash deduplication; document versioning in SQLite; user can delete and re-ingest |
| Wrong study priorities in generated plan | Medium — student focuses on wrong topics | Medium | Plan grounded in retrieved topic inventory + old exam chunks; user can manually reorder |
| User distrust of AI-generated content | Medium — user stops using the system | Medium | Source citations on every answer; explicit refusal when context is insufficient; no unexplained confidence |
| Latency on large document sets | Low–Medium — poor UX for large courses | Low for typical semester volume | Progress indicator on upload; async embedding pipeline in v2; ChromaDB query is fast at this scale |
| OpenAI API cost overrun | Low — unexpected high usage | Low for single user | Token usage logged per query; optional local embedding model (sentence-transformers) as fallback |

---

**Single highest-priority risk for v1:**
**Poor PDF text extraction.** This is both the most likely failure (many slide decks are image-heavy or use non-standard fonts) and the most impactful (if the content isn't indexed, the system cannot help). The mitigation is a clear upload-time warning and a per-page character count check, so the student knows immediately when a file is not usable. OCR support in v2 resolves the underlying issue.


## 16. Evaluation Plan

### Objective
Define how you will verify that the product is useful and trustworthy.

### Task
Create an evaluation framework for:
- retrieval quality
- factual correctness
- answer usefulness
- summary quality
- quiz quality
- study-plan relevance
- user trust / transparency

Define:
- evaluation scenarios
- success criteria
- manual or automatic checks
- feedback loop for improvement

### Deliverable
State how you will know whether the MVP is actually good enough to keep building on.


# Evaluation Plan

## Evaluation framework

### Retrieval quality
- **Metric:** Hit rate — does the correct source chunk appear in the top-8 results?
- **Method:** Manually create 20–30 question/answer pairs from uploaded materials; check whether the answer chunk is retrieved
- **Success criterion:** Hit rate > 80% on the test set
- **Check type:** Automated (script compares retrieved chunk IDs against ground-truth chunk IDs)

### Factual correctness
- **Metric:** Answer accuracy — does the answer match the source content?
- **Method:** Manual review of 20 sampled Q&A pairs; compare answer to source document
- **Success criterion:** < 5% answers contain factually incorrect statements
- **Check type:** Manual (no automated fact-checker in v1)

### Answer usefulness
- **Metric:** Thumbs up/down rating per answer (stored in SQLite)
- **Method:** In-app feedback button; weekly review of low-rated answers
- **Success criterion:** > 75% of rated answers receive a positive rating
- **Check type:** User feedback + periodic manual review

### Summary quality
- **Metric:** Does the summary cover the main topics of the source document?
- **Method:** Manual checklist — pick 3 documents, verify summary mentions top 5 topics identified by manual review
- **Success criterion:** All 5 topics covered in summary for each test document
- **Check type:** Manual

### Quiz quality
- **Metric:** Are questions answerable from the material? Are distractors plausible?
- **Method:** Manually review 20 generated questions; flag trivial, unanswerable, or trick questions
- **Success criterion:** < 10% of questions flagged as low quality
- **Check type:** Manual

### Study plan relevance
- **Metric:** Does the plan include all major topics from the course?
- **Method:** Compare generated plan topics against a manually created topic list for one course
- **Success criterion:** > 85% of major topics mentioned in the plan
- **Check type:** Manual

### User trust / transparency
- **Metric:** Does every answer include at least one source citation?
- **Method:** Automated check on 100 consecutive answers
- **Success criterion:** 100% citation coverage; 0 answers without sources
- **Check type:** Automated

---

## Feedback loop
- Thumbs down ratings trigger a log entry with the query, retrieved chunks, and answer
- Weekly review of low-rated answers identifies retrieval failures vs. generation failures
- Retrieval failures → adjust chunk size, overlap, or similarity threshold
- Generation failures → improve system prompt or grounding instructions

---

**How to know if the MVP is good enough to keep building on:**
The MVP passes the bar when: (1) retrieval hit rate exceeds 80%, (2) a student can upload all materials for one course and get a sensible study plan within 2 minutes, and (3) at least one person uses it for an actual exam and reports it was useful. Metric (3) is the real gate — the system has to survive contact with a real exam period, not just a controlled test set.


## 17. Delivery Roadmap

### Objective
Create a realistic build order.

### Task
Define three implementation phases:
- **Phase 1: MVP**
- **Phase 2: product improvement**
- **Phase 3: advanced assistant behavior**

For each phase, define:
- goals
- components
- expected improvement
- what must be stable before moving on

### Deliverable
Explain why this sequence reduces implementation risk.


# Delivery Roadmap

## Phase 1 — MVP (Weeks 1–4)

**Goal:** Working RAG pipeline with Q&A and basic summarization, usable for one real exam.

**Components:**
- File ingestion: PyMuPDF, python-pptx, plain text reader
- Chunking: RecursiveCharacterTextSplitter (512 tokens, 64 overlap)
- Embedding: OpenAI text-embedding-3-small
- Vector store: ChromaDB (local, persistent)
- Metadata + document registry: SQLite
- Q&A mode: retrieval + GPT-4o + source citations
- Summarization mode: multi-chunk summarization
- Interface: Streamlit (file upload, chat, source display)
- Deduplication: SHA-256 chunk hash
- Cosine threshold refusal

**Must be stable before moving on:**
- Q&A hit rate > 80% on a test set of 20 questions
- No hallucinations in source-cited answers
- ChromaDB persists correctly across app restarts
- At least one full course successfully ingested and queryable

---

## Phase 2 — Product improvement (Weeks 5–8)

**Goal:** Add the features that make the system feel like a study assistant, not just a search engine.

**Components:**
- Study plan mode (grounded in topic inventory + old exam analysis)
- Quiz / flashcard mode (in-app MC quiz + Anki CSV export)
- Quiz result tracking and weak-topic scoring (SQLite)
- Course-level metadata filtering in all queries
- Incremental re-indexing on file update
- Summary caching in SQLite
- Obsidian Markdown export
- Upload-time warning for text-sparse PDFs

**Expected improvement:** The system now adapts to the student's knowledge state (weak topics surfaced in quiz and plan) and produces portable outputs (Anki, Obsidian).

**Must be stable before moving on:**
- Quiz scoring and weak-topic detection produce plausible results on a real course dataset
- Anki CSV import works without errors in Anki desktop

---

## Phase 3 — Advanced assistant behavior (Weeks 9+)

**Goal:** Make the system proactive, multi-course-aware, and resilient to low-quality inputs.

**Components:**
- OCR for scanned PDFs (Tesseract or cloud OCR)
- Cross-encoder or Cohere reranker for improved retrieval
- AnkiConnect live sync
- Automatic folder watcher (new files auto-ingested on save)
- Multi-subject exam prioritization (across courses, ranked by exam date + weakness)
- Progress dashboard (topic coverage, quiz history, revision flags)
- Optional: FastAPI + lightweight frontend to replace Streamlit

---

**Why this sequence reduces implementation risk:**
Phase 1 validates the core assumption: that RAG over a student's own materials produces useful answers. If retrieval quality is poor or the chunking strategy fails, it is better to discover this with a minimal system than after building quiz logic and Anki integration on top of a broken foundation. Phase 2 adds value multipliers (personalization, exports) only after the core is proven. Phase 3 introduces architectural complexity (OCR, reranker, folder watcher) only after the product has been used in at least one real exam cycle.


## 18. Final Architecture Decision Log

### Objective
Force final clarity.

### Task
Fill in the final decision log:

- Product focus:
- Primary user:
- MVP core feature:
- Primary framework:
- Primary LLM:
- Embedding model:
- Vector database:
- Metadata / progress store:
- Interface:
- Obsidian role:
- Flashcard solution:
- Biggest excluded feature from v1:
- Main risk:
- Main success criterion:

### Deliverable
Write a short final statement:
**Why is this architecture appropriate for a professional first version of a personal AI Study Agent?**

# Final Architecture Decision Log

- **Product focus:** Personal exam preparation assistant for university students
- **Primary user:** Single student, one machine, multiple courses per semester
- **MVP core feature:** Document Q&A with source citations
- **Primary framework:** LangChain (for RAG chain orchestration) + Streamlit (interface)
- **Primary LLM:** GPT-4o (OpenAI API)
- **Embedding model:** text-embedding-3-small (OpenAI API, 1536 dimensions)
- **Vector database:** ChromaDB (local, persistent, one collection per course)
- **Metadata / progress store:** SQLite (documents, summaries, quiz results, topic progress, settings)
- **Interface:** Streamlit (chat UI with sidebar for file upload and course selection)
- **Obsidian role:** Output only — generated summaries and study plans exported as Markdown on explicit user action
- **Flashcard solution:** In-app generation via GPT-4o + Anki CSV export; AnkiConnect in v2
- **Biggest excluded feature from v1:** OCR for scanned PDFs and handwritten notes
- **Main risk:** Image-heavy lecture slides yielding no extractable text, leaving large portions of course material unindexed
- **Main success criterion:** A student can upload all materials for one course and receive a correct, source-grounded answer to a factual exam question within 8 seconds

---

**Why is this architecture appropriate for a professional first version of a personal AI Study Agent?**

This architecture is appropriate because every component choice is justified by the constraints of a solo, local, single-user product at MVP stage. ChromaDB and SQLite eliminate infrastructure overhead without sacrificing correctness or persistence. GPT-4o and text-embedding-3-small are proven, well-documented, and cost-effective at personal usage volumes. The Streamlit interface surfaces all core capabilities without frontend development investment. The strict grounding strategy (system prompt + cosine threshold + source citations) ensures the system fails visibly and honestly rather than silently and wrongly — which is the correct failure mode for a study tool a student depends on before an exam. The architecture is not over-engineered: it can be built by one developer in four weeks, used in a real exam period, evaluated against clear criteria, and extended incrementally. That is the right definition of professional for a first version.


## Closing Note

A strong design is not the one with the most components.  
It is the one with:
- clear product scope
- clear responsibilities
- realistic implementation boundaries
- testable quality
- a path from MVP to improvement

Use this notebook to make explicit choices you can justify in a professional context.
